# MJX 02 — Rendering, Cameras & Contacts

### Lab Description

Rendering is how you debug a robot model and present results. This lab gives you precise control over MuJoCo's offscreen renderer: choosing **cameras**, toggling **geom groups** (visual meshes vs. collision shapes), and visualizing **contacts**.

A key practical point: every geom belongs to a *group*. Visual meshes and collision shapes usually live in different groups, and rendering the wrong group is exactly why robots can look broken (e.g. a translucent collision sphere covering the real mesh). You'll see and fix this directly.

#### Recommended Hardware
AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)
#### Software Environment
OS: Ubuntu 24.04.4 LTS \
Install [AUP learning cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=max-pro-395). After installing AUP Learning Cloud you will have the ROCm + JAX/MJX environment (the `auplc-mujoco-mjx` course image) that this notebook is built for.

## Goals
- Render a scene from named cameras and the free camera
- Use `MjvOption.geomgroup` to show visual vs. collision geometry
- Enable contact-point / contact-force visualization
- Produce an offscreen contact video on the GPU

### Import libraries

As in MJX 01, set the EGL backend before importing `mujoco`.

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import numpy as np
import matplotlib.pyplot as plt
import mujoco
import imageio

### Build a scene with cameras, geom groups, and a falling body

This MJCF defines two **named cameras**, a body carrying *both* a visual box (group 1) and a translucent collision sphere (group 0), and a green box that falls to generate **contacts**. `mj_forward` computes positions so we can render the initial frame.

In [ ]:
# A scene with: two named cameras, a body carrying BOTH a visual geom (group 1)
# and a collision geom (group 0), and a falling box to generate contacts.
MJCF = """
<mujoco model="render_demo">
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <camera name="cam_front" pos="0 -2.2 1.0" xyaxes="1 0 0 0 0.5 1"/>
    <camera name="cam_side" pos="2.2 0 1.0" xyaxes="0 1 0 -0.5 0 1"/>
    <geom name="floor" type="plane" size="3 3 0.1" rgba="0.8 0.9 0.8 1"/>
    <body name="dual" pos="-0.5 0 0.5">
      <geom name="vis" type="box" size="0.1 0.1 0.1" group="1" rgba="0.9 0.2 0.2 1"/>
      <geom name="col" type="sphere" size="0.17" group="0" rgba="0.2 0.2 0.9 0.4"/>
    </body>
    <body name="faller" pos="0.6 0 1.2">
      <freejoint/>
      <geom name="fall" type="box" size="0.12 0.12 0.12" rgba="0.2 0.7 0.3 1"/>
    </body>
  </worldbody>
</mujoco>
"""

model = mujoco.MjModel.from_xml_string(MJCF)
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)
renderer = mujoco.Renderer(model, height=320, width=320)

### Render from named cameras

Passing `camera="cam_front"` (or `"cam_side"`) to `update_scene` renders the scene from that viewpoint. Here we render both and show them side by side.

In [ ]:
# Named cameras: render the same scene from two viewpoints.
imgs = {}
for cam in ["cam_front", "cam_side"]:
    renderer.update_scene(data, camera=cam)
    imgs[cam] = renderer.render()

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (name, img) in zip(axes, imgs.items()):
    ax.imshow(img); ax.set_title(name); ax.axis("off")
plt.tight_layout(); plt.show()

### Toggle geom groups (visual vs. collision)

`MjvOption.geomgroup` is a per-group on/off switch. The default shows all groups (so you see the collision sphere *and* the visual box). Setting only group 1 shows the **visual** geometry alone — the trick used throughout the course to render clean robot meshes.

In [ ]:
# Geom groups: the body 'dual' has a red visual box (group 1) and a translucent
# blue collision sphere (group 0). Toggling groups changes what you see.
opt_all = mujoco.MjvOption()  # default: groups 0,1,2 visible

opt_visual = mujoco.MjvOption()
opt_visual.geomgroup[:] = 0
opt_visual.geomgroup[1] = 1  # visual geoms only

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, (title, opt) in zip(axes, [("all groups (visual+collision)", opt_all), ("visual group only", opt_visual)]):
    renderer.update_scene(data, camera="cam_front", scene_option=opt)
    ax.imshow(renderer.render()); ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

### Visualize contacts

Enabling the `mjVIS_CONTACTPOINT` and `mjVIS_CONTACTFORCE` flags overlays contact markers. We drop the green box onto the floor and render the impact to an mp4.

In [ ]:
# Contacts: let the green box fall onto the floor and visualize contact points.
os.makedirs("output/videos", exist_ok=True)
contact_opt = mujoco.MjvOption()
contact_opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTPOINT] = True
contact_opt.flags[mujoco.mjtVisFlag.mjVIS_CONTACTFORCE] = True

mujoco.mj_resetData(model, data)
frames = []
for i in range(500):
    mujoco.mj_step(model, data)
    if i % 4 == 0:
        renderer.update_scene(data, camera="cam_front", scene_option=contact_opt)
        frames.append(renderer.render())

video_path = "output/videos/mjx02_contacts.mp4"
imageio.mimsave(video_path, frames, fps=30)
print(f"Saved {len(frames)} frames to {video_path}")

### Watch the contact video

In [ ]:
from IPython.display import Video
Video(url="output/videos/mjx02_contacts.mp4")

## Conclusions

You now control cameras, geom groups, and contact visualization — the tools needed to render any robot correctly and to debug physics. MJX 03 uses the visual-group trick on real robot arms while we add joint control and inverse kinematics.

---

Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT